# Chop the audios to word tensors

In [74]:
import torch
import torchaudio


# --- Energy-based silence detection ---
def split_into_word_tensors(
    waveform_name,
    frame_ms=20,           # analysis window size
    silence_thresh=0.022,  # amplitude below this = silence (tuned for yes_no.wav)
    min_silence_ms=100,    # how long a silence must last to count as a word gap
    min_word_ms=100,       # discard any "word" shorter than this (filters noise blips)
    padding_ms=50,         # keep a little audio padding around each word
):
    # --- Load audio ---
    waveform, sample_rate = torchaudio.load(waveform_name)

    # Convert stereo -> mono if needed
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    frame_len = int(sample_rate * frame_ms / 1000)
    signal = waveform[0]  # mono channel, shape: (num_samples,)

    num_frames = signal.shape[0] // frame_len
    energies = torch.tensor([
        signal[i * frame_len : (i + 1) * frame_len].abs().mean()
        for i in range(num_frames)
    ])

    is_silent = energies < silence_thresh
    min_silence_frames = int(min_silence_ms / frame_ms)
    min_word_frames = int(min_word_ms / frame_ms)

    word_ranges = []
    start = None
    silence_run = 0

    for i, silent in enumerate(is_silent):
        if not silent:
            if start is None:
                start = i
            silence_run = 0
        else:
            silence_run += 1
            if start is not None and silence_run >= min_silence_frames:
                end = i - silence_run + 1
                if end - start >= min_word_frames:   # drop too-short blips
                    word_ranges.append((start, end))
                start = None

    if start is not None and num_frames - start >= min_word_frames:
        word_ranges.append((start, num_frames))

    # Convert frame indices -> sample indices, with padding, and slice tensors
    padding_samples = int(sample_rate * padding_ms / 1000)
    word_tensors = []
    for start_frame, end_frame in word_ranges:
        start_sample = max(0, start_frame * frame_len - padding_samples)
        end_sample = min(signal.shape[0], end_frame * frame_len + padding_samples)
        word_tensors.append(waveform[:, start_sample:end_sample])

    return word_tensors, sample_rate

word_tensors_yes, sample_rate = split_into_word_tensors("yes_new_38.wav")
print(len(word_tensors_yes))

word_tensors_no, sample_rate = split_into_word_tensors("no_new_42.wav")
print(len(word_tensors_no))

print(word_tensors_yes[0].shape)
print(word_tensors_no[0].shape)


38
42
torch.Size([1, 13230])
torch.Size([1, 26460])


# Reduce sequence lengths using MelSpectrogram

In [75]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=sample_rate,
    n_fft=1024,        # window size for each FFT frame
    hop_length=256,     # step between frames — controls time resolution
    n_mels=40,           # number of mel frequency bins (your feature dimension)
)
amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

def extract_mel(word_tensor):
    mel = mel_transform(word_tensor)          # shape: (1, n_mels, T_frames)
    mel_db = amplitude_to_db(mel)              # log-scale, same shape
    return mel_db.squeeze(0).T                 # shape: (T_frames, n_mels)

mel_yes = [extract_mel(w) for w in word_tensors_yes]   # each: (T_frames, 40)
mel_no = [extract_mel(w) for w in word_tensors_no]

# Split the dataset into train, validation and test dataset

In [76]:
from sklearn.model_selection import train_test_split
train_yes, temp = train_test_split(mel_yes, test_size=0.2, random_state=42)
validation_yes, test_yes = train_test_split(temp, test_size=0.5, random_state=42)

train_no, temp = train_test_split(mel_no, test_size=0.2, random_state=42)
validation_no, test_no = train_test_split(temp, test_size=0.5, random_state=42)

# Set the train, validation and test labels

In [77]:
train_yes_labels = []
for i in range(len(train_yes)):
    train_yes_labels.append(0)
validation_yes_labels = []
for i in range(len(validation_yes)):
    validation_yes_labels.append(0)
test_yes_labels = []
for i in range(len(test_yes)):
    test_yes_labels.append(0)

train_no_labels = []
for i in range(len(train_no)):
    train_no_labels.append(1)
validation_no_labels = []
for i in range(len(validation_no)):
    validation_no_labels.append(1)
test_no_labels = []
for i in range(len(test_no)):
    test_no_labels.append(1)



# Combine the training, validation and test data

In [78]:
train_data = train_yes + train_no
validation_data = validation_yes + validation_no
test_data = test_yes + test_no

train_labels = train_yes_labels + train_no_labels
validation_labels = validation_yes_labels + validation_no_labels
test_labels = test_yes_labels + test_no_labels

# Normalize across the dataset

In [79]:
all_train = torch.cat([x for x in train_data], dim=0)  # (total_frames, 40)

mean, std = all_train.mean(0), all_train.std(0) + 1e-6

def normalize(x):
    return (x - mean) / std

train_data = [normalize(x) for x in train_data]
validation_data = [normalize(x) for x in validation_data]
test_data = [normalize(x) for x in test_data]

# Create the machine learning model (GRU)

In [80]:
import torch
import torch.nn as nn
class SimpleRNNModel(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, output_size)

    def forward(self, X):
        outputs, last_state = self.rnn(X)
        return self.output(outputs[:, -1, :])  # Use the last output for prediction

torch.manual_seed(42)
model = SimpleRNNModel(input_size=40, hidden_size=32, output_size=1).to("cuda")

# Define and do the training

In [81]:
import random

def train_sgd(model, optimizer, criterion, X_train, y_train, n_epochs):

    n = len(X_train)
    indices = list(range(n))

    for epoch in range(n_epochs):

        epoch_loss = 0.0

        random.shuffle(indices)
        for i in indices:

            # Get one training example
            X = X_train[i].unsqueeze(0).to("cuda")
            y = y_train[i].view(1, 1)

            # Forward pass
            y_pred = model(X)

            #print(y_pred.shape)
            #print(y.shape)

            # Calculate loss
            loss = criterion(y_pred, y)

            # Backpropagation
            loss.backward()

            # Update parameters
            optimizer.step()

            # Clear gradients
            optimizer.zero_grad()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / n
        print(f"Epoch {epoch + 1}/{n_epochs}, Avg Loss: {avg_loss}")

learning_rate = 0.001
xentropy = nn.BCEWithLogitsLoss()
n_epochs = 10


train_labels = torch.tensor(train_labels, dtype=torch.float32).to("cuda")

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

model.train()
train_sgd(model, optimizer, xentropy, train_data, train_labels, n_epochs)

Epoch 1/10, Avg Loss: 0.3201240322419575
Epoch 2/10, Avg Loss: 0.0461015963838214
Epoch 3/10, Avg Loss: 0.02254076665710835
Epoch 4/10, Avg Loss: 0.014319478903734495
Epoch 5/10, Avg Loss: 0.010130754051109156
Epoch 6/10, Avg Loss: 0.007647509738388989
Epoch 7/10, Avg Loss: 0.006023144578590753
Epoch 8/10, Avg Loss: 0.004901072697802668
Epoch 9/10, Avg Loss: 0.004081365955431783
Epoch 10/10, Avg Loss: 0.0034631317082260337


# Testing

In [82]:
import torchmetrics

def evaluate_tm(model, X_test, y_test, metric):
    model.eval()
    metric.reset()  # reset the metric at the beginning
    with torch.no_grad():
        for i in range(len(X_test)):
            X = X_test[i].unsqueeze(0).to("cuda")
            y = y_test[i].view(1, 1)

            y_pred = model(X)

            metric.update(y_pred, y)
    return metric.compute()

test_labels = torch.tensor(test_labels, dtype=torch.float32).to("cuda")

accuracy = torchmetrics.Accuracy(task="binary", num_classes=2).to("cuda")

print(evaluate_tm(model, test_data, test_labels, accuracy))


tensor(1., device='cuda:0')
